# ITCC508 Lab Exercise PT-M1: Domain-Specific RAG Chatbot

**Domain:** Nephrotic Syndrome Patient Education Q&A Chatbot

## Task 1 — Domain & Dataset Description

**Problem statement.** Nephrotic syndrome is a kidney condition that requires patients to make daily decisions about diet, fluid intake, and physical activity, often without a clinician immediately on hand to answer a quick question. Newly diagnosed patients and their families frequently have simple, recurring questions "is this food okay for me to eat," "how much water can I drink today," "am I allowed to exercise" that don't need a full doctor's visit to answer, but do need to be answered *correctly* rather than guessed at. General-purpose chatbots are a poor fit for this because they will confidently answer from broad training knowledge even when they have no real basis for the specific guidance a patient needs, which is risky in a health context. This project builds a chatbot that only answers from a small set of vetted patient-education documents, and is explicitly instructed to refuse rather than guess when a question falls outside that material.

**Why RAG specifically.** A base LLM with no retrieval step would either refuse everything (too cautious to be useful) or hallucinate plausible-sounding but ungrounded medical advice (unsafe). Retrieval-Augmented Generation solves this by forcing every answer to be built only from text chunks that were actually retrieved from our own trusted documents, and by giving the model an explicit fallback instruction for when nothing relevant is found. Task 3 later in this notebook is designed to directly demonstrate what happens when those two safeguards (grounding + fallback) are removed.

**Source files placed in `./my_data/`:**
1. `nkf_teachback.pdf` — National Kidney Foundation, *Nephrotic Syndrome: A Guide to Educate Your Patients* (a "teach-back" style patient card, meaning it's written specifically to be read back and confirmed with a patient, so its language is short, plain, and instructional — good chunking material).
2. `nkf_what_you_should_know.pdf` — National Kidney Foundation, *Nephrotic Syndrome: What You Should Know* (a patient card covering testing, treatment, dietitian referral, and lifestyle guidance such as exercise, smoking, and alcohol).
3. `nephcure_factsheet.pdf` — NephCure Kidney International, *Nephrotic Syndrome Fact Sheet* (broader background: causes, symptoms, and general disease information, written in a different style/structure from the two NKF documents above).

Using three documents rather than one is deliberate: it lets us demonstrate, in Cell 7's printed output, that the retriever is genuinely pulling relevant chunks from *multiple* source files rather than just repeatedly returning the same document, a small but meaningful proof that the vector search step is doing real semantic work rather than trivially returning "the only file available."

**Model swap note.** The lab template's default model (Groq's `llama-3.1-8b-instant`) is swapped for `openai/gpt-oss-20b`, OpenAI's own open-weight model, served through Groq's free hosting. This still runs through the exact same `ChatGroq` class and the same free `GROQ_API_KEY` from Cell 2 — the only line that changes anywhere in the whole notebook because of the model swap is the `model_name=` string inside Cell 5. Every other cell is left structurally identical to the original template, per the requirement that only the code strictly necessary to accommodate the new model may be touched. (Separately, two small dependency-version fixes were also required to get the *original, unmodified* template code to actually run on Colab's current package versions those are explained in Cell 1 below, and are unrelated to the model choice.)

### Cell 1: Environment Setup & Package Installation

**What this cell does.** Colab's runtime starts essentially empty none of the specialized libraries this pipeline depends on are pre-installed, so the very first thing the notebook must do is install them with `pip`, before any of our own code can import anything. Everything downstream (loading PDFs, generating embeddings, talking to Groq) will fail immediately with an `ImportError` or `ModuleNotFoundError` if this cell hasn't been run first.

**Why each package is needed:**
- **`langchain`** — the core orchestration library. On its own it doesn't talk to any specific vector store or LLM provider; instead it defines the common interfaces (chains, prompts, retrievers) that let all the other, more specific packages below plug into one unified pipeline. Cell 6 in particular depends directly on functions that live inside this package.
- **`langchain-core`** — the lower-level package that `langchain` itself, and every provider-specific integration package (`langchain-groq`, `langchain-huggingface`, `langchain-community`), are built on top of. It defines shared building blocks like `ChatPromptTemplate` (used in Cell 5) and the internal message format that gets passed between the prompt, the retriever, and the LLM. Because so much depends on it, keeping its version compatible with everything else in this list turned out to matter a lot in practice (see the version-pinning note further down).
- **`langchain-community`** — a large collection of community-maintained integrations that aren't central enough to live in `langchain` itself. We specifically need it for `DirectoryLoader` (Cell 3, which reads our three PDFs off disk) and the `Chroma` vector store wrapper (Cell 4).
- **`langchain-text-splitters`** — provides `RecursiveCharacterTextSplitter` (Cell 3), the tool that breaks each loaded PDF into small, overlapping text chunks. This used to ship inside `langchain` itself but is now its own package.
- **`langchain-groq`** — the specific integration that lets LangChain talk to Groq's inference API. This is what makes `ChatGroq` (Cell 5) work at all, and is exactly what allows us to point at Groq's hosted build of `openai/gpt-oss-20b` instead of only Groq's own Llama models.
- **`langchain-huggingface`** — provides `HuggingFaceEmbeddings` (Cell 4), the wrapper that lets us download and run the open-source `all-MiniLM-L6-v2` sentence-embedding model locally inside the Colab runtime, entirely for free and without calling any external embedding API.
- **`chromadb`** — the actual vector database engine underneath LangChain's `Chroma` wrapper. This is what physically stores every chunk's embedding vector in memory and performs the nearest-neighbor similarity search when the retriever is queried in Cell 7 onward.
- **`pypdf`** — a lightweight, pure-Python PDF text-extraction library. `DirectoryLoader` uses this (or `unstructured`, below) under the hood to actually pull readable text out of each `.pdf` file in `my_data/`.
- **`unstructured`** — a more heavyweight, general-purpose document parser that `DirectoryLoader` can also fall back on, particularly useful for messier or more complex document layouts than a simple PDF.

**Two fixes that were required to get the template running as-is:**

1. **`unstructured[pdf]` extra.** Installing the base `unstructured` package alone is *not* enough to parse PDF files with it the PDF-specific parsing code path (`partition_pdf()`) depends on extra libraries (like `pdfminer.six`) that are only pulled in if you explicitly request the `[pdf]` extra. Without this, Cell 3 throws an `ImportError` the moment `DirectoryLoader` tries to read our first PDF. This has nothing to do with which LLM we're using every student loading PDF files with this exact template needs this fix, regardless of model choice.

2. **Pinning the whole LangChain package family below version 1.0.** In its 1.0 release, LangChain restructured how the library is organized: the specific functions `create_retrieval_chain` and `create_stuff_documents_chain` (both used in Cell 6, exactly as written in the original template) were moved out of `langchain.chains` and into a new, separate `langchain-classic` package. On top of that, `langchain-core`'s internal message-handling format also changed in a way that isn't backward-compatible with the older `langchain`/`langchain-community` releases. The practical effect: if you `pip install langchain` with no version constraint, Colab grabs the newest release by default, and Cell 6's import statements copied verbatim from the lab template — simply no longer resolve. Pinning *only* `langchain` and `langchain-community` below `1.0` isn't sufficient either, because pip will still separately grab the newest, incompatible `langchain-core`, `langchain-groq`, and `langchain-huggingface` to go with them. The fix used here pins every LangChain-family package below `1.0` **together** in the same install command, so pip's dependency resolver is forced to settle on one consistent, mutually-compatible set of versions across all of them. This keeps every import statement in Cells 5 and 6 working exactly as the template originally specified them, with zero code changes needed in those cells themselves. Again, this is purely a packaging/versioning issue affecting anyone running this template on a fresh Colab runtime today it is not something introduced by, or specific to, our choice of `openai/gpt-oss-20b`.

`-q` on each `pip install` simply suppresses the normal installation log output so the notebook's saved output stays short and readable, rather than filling several screens with progress bars every time this cell runs. Aside from the two dependency fixes explained above.

In [12]:
# Cell 1: Environment Setup & Package Installation
!pip install -q "langchain<1.0" "langchain-core<1.0" "langchain-community<1.0" "langchain-text-splitters<1.0" "langchain-groq<1.0" "langchain-huggingface<1.0" chromadb pypdf unstructured
# NOTE: base "unstructured" install is missing PDF-parsing extras (pdfminer, etc.),
!pip install -q "unstructured[pdf]"


### Cell 2: Secure API Key Configuration

**What this cell does.** Before we can call any model hosted on Groq, LangChain's `ChatGroq` class needs a valid Groq API key. This cell asks for that key once, securely, and stores it where the rest of the notebook can find it automatically.

**Line-by-line:**
- `import os` — the standard library module for interacting with the operating system, including reading and writing **environment variables**. Environment variables are a common way to pass secrets (like API keys) into a program without hard-coding them directly into source code.
- `import getpass` — a standard library module whose `getpass()` function displays a masked input prompt (similar to a password field) so that whatever you type is never echoed back visibly on screen or saved into the notebook's output history. This matters specifically for the "basic API-key protection practices" discussion your IEEE report's Conclusion section is required to cover: even a free-tier API key shouldn't end up sitting in plain text inside a `.ipynb` file that might later get shared, committed to GitHub, or copied into a report.
- `if "GROQ_API_KEY" not in os.environ:` — checks whether the key has already been set in this runtime's environment before asking for it again. This means if you re-run this cell later in the same session (for example, after Colab briefly disconnects and reconnects, or if you re-run cells out of order), you won't be interrupted with a repeat prompt as long as the same runtime session is still active.
- `os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")` — actually prompts for the key and stores it into the environment variable `GROQ_API_KEY`. `ChatGroq` (initialized later in Cell 5) is written to automatically look for a variable with exactly this name, so once it's set here, we never have to reference the key directly again anywhere else in the notebook.


In [13]:
# Cell 2: Secure API Key Configuration
import os
import getpass
# Prompts for API key securely without saving or echoing plain text in notebook cells
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")


### Cell 3: Loading Custom Data & Chunking

**What this cell does.** This is the first stage of the actual RAG pipeline: it locates our three nephrotic-syndrome PDFs on disk, reads their text content into memory, and then breaks that text down into small, manageable pieces ("chunks") that are the right size to later be embedded and searched individually rather than treating each entire PDF as one giant, unsearchable block of text.

**Line-by-line:**
- `from langchain_community.document_loaders import DirectoryLoader` — imports a loader class capable of scanning an entire folder and loading every matching file inside it, rather than requiring us to write a separate loading call for each of our three individual PDFs by hand.
- `from langchain_text_splitters import RecursiveCharacterTextSplitter` — imports the specific text splitter we'll use. "Recursive" here means it tries to split on natural boundaries first (paragraph breaks, then sentence breaks, then words) before falling back to a hard character cutoff, which produces cleaner, more semantically coherent chunks than splitting at a fixed character count blindly.
- `os.makedirs("my_data", exist_ok=True)` — creates a folder named `my_data` in the current working directory if it doesn't already exist. `exist_ok=True` means this line won't raise an error if the folder is already there (for example, on a re-run). This is the exact folder our three PDFs (`nkf_teachback.pdf`, `nkf_what_you_should_know.pdf`, `nephcure_factsheet.pdf`) need to be uploaded into via Colab's file browser *before* this cell finishes running.
- `loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)` — configures the loader to look inside `my_data/`, matching any file at any nesting depth (`**/*.*`), and to display a progress bar while it works (`show_progress=True`), which is useful feedback when loading multiple files.
- `raw_documents = loader.load()` — actually executes the load, reading every matching file's text content into a list of LangChain `Document` objects. At this point we have three `Document` objects — one whole, unsplit document per PDF — each carrying its extracted text plus metadata such as the originating filename.
- `text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` — configures the splitter with two key parameters. `chunk_size=500` caps each resulting chunk at roughly 500 characters, which is small enough that a single chunk usually covers one focused idea (e.g. "the fluid restriction guidance" rather than "the entire dietitian section plus the exercise section plus the medication section"). `chunk_overlap=50` means each chunk shares its last 50 characters with the start of the next chunk, so a sentence or idea that happens to fall right on a chunk boundary isn't split away from its surrounding context entirely.
- `documents = text_splitter.split_documents(raw_documents)` — applies the splitter to all three raw documents at once, producing the final flat list of small chunks that Cell 4 will embed. A single 2-page PDF might become 8–15 chunks after this step, for example.
- `print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")` — a simple sanity-check line confirming both numbers at a glance. This printed output is worth screenshotting directly into your report's Implementation Details table, since it's concrete evidence of how your specific dataset was processed.


In [14]:
# Cell 3: Loading Custom Data & Chunking
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create target data directory
os.makedirs("my_data", exist_ok=True)

# Load all documents from the directory
loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)
raw_documents = loader.load()

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
documents = text_splitter.split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")


100%|██████████| 3/3 [00:07<00:00,  2.64s/it]


Loaded 3 raw document(s) and split into 25 chunks.


### Cell 4: Embedding Model & Vector DB Indexing

**What this cell does.** This is the stage that turns our text chunks into something a computer can actually search by *meaning* rather than by exact keyword match. Every chunk from Cell 3 gets converted into a numeric vector (an "embedding"), and all of those vectors are stored inside an in-memory ChromaDB database, ready to be searched the moment a user asks a question.

**Line-by-line:**
- `from langchain_huggingface import HuggingFaceEmbeddings` — imports the wrapper class that lets us run an open-source embedding model locally, entirely for free, without needing any external embedding API or its own separate API key.
- `from langchain_community.vectorstores import Chroma` — imports LangChain's wrapper around the ChromaDB vector database engine (installed in Cell 1), giving us a simple, high-level interface for storing and later querying embedding vectors.
- `embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` — downloads and loads a compact, well-established open-source sentence-embedding model. Conceptually, this model reads a piece of text and outputs a fixed-length list of numbers (a vector) positioned in a high-dimensional space such that chunks with *similar meaning* end up with vectors that are mathematically close together even if they don't share many of the same exact words. This is what makes semantic search possible: a user asking "can I drink a lot of water" can match a chunk that talks about "fluid restriction" even though neither phrase shares the words "drink" or "water."
- `vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)` — runs every chunk produced in Cell 3 through the embedding model, and stores the resulting vector, alongside the original chunk text and its source-file metadata, inside a new in-memory ChromaDB collection. "In-memory" means this vector store lives only in the current Colab runtime's RAM and disappears when the runtime resets for a lab exercise that's fine, since we intentionally rebuild it fresh from our three source PDFs every time the notebook is run top to bottom.
- `retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` — wraps the raw vector store in a "retriever" interface, which is the specific shape of object that LangChain's chain-building functions (Cell 6) expect to plug into a RAG pipeline. `search_kwargs={"k": 3}` configures it to return the **3 most similar chunks** for any given query. `k=3` is a deliberate balance: too small a `k` risks missing a relevant chunk that happened to score just below the cutoff, while too large a `k` risks diluting the LLM's context window with less-relevant filler text, which can actually make answers *less* focused.

In [15]:
# Cell 4: Embedding Model & Vector DB Indexing
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize open-source embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store embeddings into Chroma vector database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# Set vectorstore as a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Cell 5: Model Initialization and Domain System Prompt

**What this cell does.** This is where the actual language model is configured, and where the rules it must follow when generating answers are defined. Two things get created here: the `llm` object (the model itself) and the `prompt` object (the instructions plus a template slot for retrieved context, which Cell 6 will plug the retriever's output into).

**Line-by-line:**
- `from langchain_groq import ChatGroq` — **unchanged import.** `openai/gpt-oss-20b` is hosted *by Groq*, not by OpenAI directly — it's OpenAI's own openly-released, open-weight model, made available through Groq's fast inference infrastructure. Because Groq is still the service actually serving the model, we keep using the exact same `ChatGroq` integration class rather than switching to a different LangChain provider package.
- `from langchain_core.prompts import ChatPromptTemplate` — unchanged import, used to build the structured, multi-turn prompt template below.
- **`model_name="openai/gpt-oss-20b"`** — **this is the single line changed for the entire model swap.** The original template specified `"llama-3.1-8b-instant"` here. Because the change is scoped to just this one string, and both models are served through the identical `ChatGroq` class using the identical `GROQ_API_KEY`, this satisfies the "change only what the swap requires" rule about as narrowly as possible nothing about how the model is called, authenticated, or wired into the rest of the pipeline needed to change, only *which* model string is requested.
- `temperature=0` — kept at 0 for this baseline configuration. Temperature controls how much randomness is injected into the model's token-by-token word choices during generation: `0` makes the model as close to fully deterministic as possible, consistently picking its highest-confidence next word rather than sampling more creatively. For a grounded, factual Q&A chatbot, low temperature is what we want it keeps answers tightly anchored to what the retrieved context actually says, rather than "improvising" phrasing or content. This value gets deliberately raised to `1.0` later, but only inside the separate Task 3 stress-test cells, never here.
- `system_prompt = (...)` — defines the instruction block that will always be sent to the model *before* the user's actual question. Three things are happening in this string: (1) it tells the model it is a specialized assistant scoped to "the user's uploaded domain" deliberately generic wording so the same prompt works for any domain, not just ours; (2) it instructs the model to answer **strictly** using only the `{context}` that gets filled in later, rather than drawing on its own general training knowledge; and (3) it gives an exact, fixed fallback sentence to use whenever the answer isn't present in that context. This fallback instruction is the single most important safeguard being tested in Task 2 and Task 3 later in this notebook it's the difference between a system that admits when it doesn't know something and one that guesses.
- `prompt = ChatPromptTemplate.from_messages([...])` — assembles the final prompt template out of two message "turns": a `"system"` turn containing the instructions above (with `{context}` left as an unfilled placeholder), and a `"human"` turn containing another placeholder, `{input}`, for whatever the user actually asks. Cell 6's chain-building functions are what actually fill in both of these placeholders at query time  `{context}` gets replaced with the retriever's top-3 chunks, and `{input}` gets replaced with the live user question.

In [16]:
# Cell 5: Model Initialization and Domain System Prompt
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialize the SLM
# NOTE: model swapped from "llama-3.1-8b-instant" to Groq's hosted OpenAI open-weight model
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
)

# Custom domain system prompt
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])


### Cell 6: Pipeline Assembly

**What this cell does.** Everything built in the previous cells — the retriever (Cell 4), and the prompt template plus LLM (Cell 5) gets wired together here into a single, callable `rag_chain` object. After this cell runs, answering a question is as simple as calling `rag_chain.invoke(...)` once, and every step (retrieval, prompt-filling, generation) happens automatically behind that one call.

**Line-by-line:**
- `from langchain.chains import create_retrieval_chain` — imports the top-level function that will assemble the full end-to-end RAG pipeline.
- `from langchain.chains.combine_documents import create_stuff_documents_chain` — imports the function that builds the specific sub-chain responsible for combining retrieved documents with the prompt and sending the result to the LLM.
- `combine_docs_chain = create_stuff_documents_chain(llm, prompt)` — builds a chain that, given a list of retrieved `Document` chunks, concatenates ("stuffs") all of their text together into the `{context}` slot of our prompt template from Cell 5, and then sends the fully-filled prompt to `llm` for a response. "Stuff" is the standard term in LangChain for this simplest possible context-combination strategy as opposed to more elaborate strategies like map-reduce or refine, which are better suited to much larger document sets than the handful of small chunks we're working with here.
- `rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` — wraps the `combine_docs_chain` together with our `retriever` from Cell 4 into one final, complete chain. Calling `rag_chain.invoke({"input": some_question})` on the resulting object now automatically performs all three RAG steps in sequence: (1) embed the incoming question and retrieve the top-3 most similar chunks from ChromaDB, (2) stuff those chunks into the `{context}` slot of the prompt alongside the question in the `{input}` slot, and (3) send the completed prompt to `openai/gpt-oss-20b` via Groq and return its generated answer.

In [17]:
# Cell 6: Pipeline Assembly
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Combine prompt and LLM to process context
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)


### Cell 7: Testing Your Custom Domain Chatbot (In-Domain Query)

**What this cell does.** This is the first real end-to-end test of the assembled pipeline: a genuine, in-domain question is sent through `rag_chain`, and both the generated answer and the specific source chunks that were used to produce it are printed out for inspection.

**Line-by-line:**
- `user_query = "Summarize the key information found in the documents."` — a deliberately broad question, chosen because it's clearly in-domain (it directly references "the documents," i.e. our uploaded PDFs) and should be answerable using content pulled from across all three of our nephrotic syndrome source files, rather than just one narrow fact.
- `response = rag_chain.invoke({"input": user_query})` — runs the complete pipeline in one call: embeds `user_query`, retrieves the top-3 matching chunks from ChromaDB, stuffs them into the prompt alongside the question, and sends the result to `openai/gpt-oss-20b`. The returned `response` is a dictionary containing several keys, two of which we use immediately below.
- `response["answer"]` — the model's actual generated answer text, which we print directly.
- `response["context"]` — not just the answer, but the *actual list of retrieved `Document` objects* that were fed into the prompt to produce it. This is extremely useful for transparency and debugging: it lets us verify exactly what evidence the model was working from, rather than trusting the answer blindly.
- `for i, doc in enumerate(response["context"]): print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))` — loops over each of the (up to 3) retrieved chunks and prints which original PDF filename it came from, using the metadata that `DirectoryLoader` automatically attached back in Cell 3. This is the concrete evidence, mentioned back in the Task 1 write-up, that the retriever is genuinely pulling relevant material from multiple different source documents rather than always defaulting to the same single file worth including as a screenshot in your report's Experimental Results section.


In [18]:
# Cell 7: Testing Your Custom Domain Chatbot
# Test Case 1: In-Domain Query
user_query = "Summarize the key information found in the documents."
response = rag_chain.invoke({"input": user_query})

print("--- DOMAIN QUERY ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE CHUNKS ---")
for i, doc in enumerate(response["context"]):
    print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))


--- DOMAIN QUERY ANSWER ---
**Key Points from the Documents**

1. **Kidney Biopsy**
   - Used for diagnosis and to guide treatment.
   - Involves taking one or more tiny kidney samples.
   - Samples are examined under special microscopes for detailed analysis.

2. **Treatment Overview**
   - Treatment depends on the specific kidney disease and the patient’s overall health.
   - Common therapeutic strategies include:
     - **Dietary changes** (e.g., salt and fluid restriction).
     - **Medications** aimed at:
       - Reducing excess salt and fluid in the body.
       - Decreasing protein loss in the urine.
       - Lowering blood cholesterol.

3. **Common Kidney Conditions Mentioned**
   - **Focal Segmental Glomerulosclerosis (FSGS)**
   - **IgA Nephropathy**
   - **Lupus‑related kidney disease**
   - **Diabetes‑related kidney disease**
   - **Infections** that can affect the kidneys (Hepatitis B, Hepatitis C, HIV, others)

4. **Clinical Features & Symptoms**
   - Weight gain from fl

### Task 2: Out-of-Domain Fallback Test

**What this cell does.** This directly tests the fallback safeguard defined in Cell 5's system prompt. The baseline configuration is still active here `temperature=0`, and the fallback-instruction sentence is still present in the system prompt so this cell establishes what "correct" behavior looks like before we deliberately break it in Task 3.

**Design choices explained:**
- **Why this specific query.** `out_of_domain_query = "Is it safe to take ibuprofen for a headache while pregnant?"` was chosen deliberately, not arbitrarily: it's health-related, so it plausibly *sounds* like something a medically-themed chatbot might be expected to know something about, but it has no overlap at all with the actual content of our three nephrotic syndrome PDFs (none of which discuss ibuprofen, headaches, or pregnancy). This makes it a meaningful test of whether the system is truly grounded in retrieval rather than a weaker test using an obviously silly, completely unrelated topic (like asking about sports scores), which wouldn't tell us much about whether topic-adjacent but ungrounded questions are handled correctly too.
- `response_ood = rag_chain.invoke({"input": out_of_domain_query})` — runs the exact same pipeline as Cell 7, just with this new, intentionally out-of-domain question. The retriever will still return its top-3 "closest" chunks even though none of them are actually relevant (vector similarity search always returns *something*, even if the best match is still a poor one) the real test is whether the system prompt's fallback instruction successfully stops the model from using those irrelevant chunks to construct a made-up answer anyway.
- **Expected result:** `response_ood["answer"]` should come back as exactly the fixed fallback sentence defined in Cell 5: *"I cannot answer based on the provided domain data."* This exact output is what gets compared directly against Task 3's result in the next section.

In [19]:
# Task 2: Out-of-Domain Fallback Test
out_of_domain_query = "Is it safe to take ibuprofen for a headache while pregnant?"
response_ood = rag_chain.invoke({"input": out_of_domain_query})

print("--- OUT-OF-DOMAIN QUERY ---")
print(out_of_domain_query)
print("\n--- MODEL ANSWER (Baseline: temperature=0, fallback rule active) ---")
print(response_ood["answer"])


--- OUT-OF-DOMAIN QUERY ---
Is it safe to take ibuprofen for a headache while pregnant?

--- MODEL ANSWER (Baseline: temperature=0, fallback rule active) ---
I cannot answer based on the provided domain data.


### Task 3: Hallucination Stress-Test

**What these cells do, and why, step by step:**

**Step 1 — Change Parameters.** A brand-new `llm_stress` and `system_prompt_stress` are created here, deliberately separate from the original `llm`/`system_prompt` objects from Cell 5, so that Task 2's baseline configuration is left completely intact and can still be re-run or referenced afterward for comparison. Two specific changes are made, exactly as the assignment instructions specify:
- **`temperature=1.0`** (raised from `0`) — this substantially increases the randomness in the model's token sampling during generation. At `temperature=0`, the model almost always picks its single highest-probability next word at each step; at `temperature=1.0`, it samples more freely across a wider range of plausible next words, which tends to produce more varied, more "creative," and less tightly-anchored output including, potentially, output that drifts away from strictly reflecting the retrieved context and instead draws more on the model's own general training knowledge.
- **The fallback-instruction sentence is deleted from the system prompt entirely** — comparing `system_prompt_stress` against the original `system_prompt`, the sentence *"If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'"* is simply absent. Without this explicit instruction, the model has no defined fallback behavior to follow when the retrieved context doesn't actually contain a real answer it must decide for itself what to do, which is precisely the ungrounded scenario this stress-test is designed to expose.
- `combine_docs_chain_stress = create_stuff_documents_chain(llm_stress, prompt_stress)` and `rag_chain_stress = create_retrieval_chain(retriever, combine_docs_chain_stress)` — re-assemble a second, independent RAG chain using the new stress-test model and prompt, but reusing the *exact same* `retriever` object from Cell 4 without any changes. This is an important control: the vector database, the source documents, and the retrieval logic are held constant across both Task 2 and Task 3, so any difference we observe in the final answer can be attributed specifically to the temperature and system-prompt changes not to some unrelated difference in what was retrieved.

**Step 2 — Re-run Query.** The exact same `out_of_domain_query` string defined back in Task 2 is reused here unchanged, and sent through the new `rag_chain_stress` instead of the original `rag_chain`. Reusing the identical question is what makes this a fair, controlled comparison rather than an apples-to-oranges one.

**Step 3 — Observe & Compare.** With both `response_ood["answer"]` (Task 2, grounded baseline) and `response_stress["answer"]` (Task 3, ungrounded stress-test) now available, the markdown cell below prompts you to actually record and interpret the difference for your IEEE report's Experimental Results and Hallucination Analysis section specifically, whether the model still declined appropriately even without an explicit instruction to do so, or whether it began generating unsupported, "hallucinated" content instead.

In [20]:
# Task 3, Step 1: Modified model + system prompt for the stress-test
llm_stress = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=1.0
)

# Fallback-instruction sentence removed
system_prompt_stress = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n\n"
    "Context:\n{context}"
)

prompt_stress = ChatPromptTemplate.from_messages([
    ("system", system_prompt_stress),
    ("human", "{input}"),
])

# Re-assemble the chain with the stress-test model/prompt (retriever is unchanged)
combine_docs_chain_stress = create_stuff_documents_chain(llm_stress, prompt_stress)
rag_chain_stress = create_retrieval_chain(retriever, combine_docs_chain_stress)


In [21]:
# Task 3, Step 2: Re-run the exact same out-of-domain query
response_stress = rag_chain_stress.invoke({"input": out_of_domain_query})

print("--- OUT-OF-DOMAIN QUERY (same as Task 2) ---")
print(out_of_domain_query)
print("\n--- MODEL ANSWER (Stress-test: temperature=1.0, fallback rule removed) ---")
print(response_stress["answer"])


--- OUT-OF-DOMAIN QUERY (same as Task 2) ---
Is it safe to take ibuprofen for a headache while pregnant?

--- MODEL ANSWER (Stress-test: temperature=1.0, fallback rule removed) ---
I’m not able to give medical advice, but the information provided suggests that you should:

- **Talk with your healthcare provider** about taking ibuprofen during pregnancy.  
- The context notes that **NSAIDs (like ibuprofen and naproxen)** are generally avoided because they can be harmful, and that **certain precautions or adjustments to treatment might be needed** when you are pregnant.  

So, before using ibuprofen for a headache while pregnant, consult your doctor or midwife to determine if it’s safe for you.


### Step 3: Observe & Compare (write-up)

- **Baseline (Task 2 — temperature=0, fallback active):**
The model correctly refused, returning exactly the fallback sentence defined in the system prompt: "I cannot answer based on the provided domain data." This confirms the grounding safeguard worked as intended the model did not attempt to answer a question outside its retrieved context.

- **Stress-test (Task 3 — temperature=1.0, fallback removed):**
The model did not refuse. Instead of declining, it responded: "I'm not able to give medical advice, but the information provided suggests that you should: Talk with your healthcare provider about taking ibuprofen during pregnancy. The context notes that NSAIDs (like ibuprofen and naproxen) are generally avoided because they can be harmful, and that certain precautions or adjustments to treatment might be needed when you are pregnant..."

  Interestingly, the model added a disclaimer ("I'm not able to give medical advice") that wasn't part of the system prompt at all suggesting the base model has its own built-in caution around medical topics, layered on top of whatever the (missing) fallback instruction would have enforced. Despite that disclaimer, it still went on to give specific guidance framed as coming from "the context" even though none of the three source documents actually discuss ibuprofen or pregnancy.

- **Discussion — the mechanism:** At temperature=0 with the fallback instruction present, the model had both a low-randomness generation setting and an explicit rule for what to do with weak or irrelevant context refuse. Removing the fallback instruction left the model to use its own judgment about how to handle marginally-related retrieved chunks, and raising temperature to 1.0 made it more willing to extrapolate beyond what those chunks actually supported rather than sticking to the narrowest safe response.